# HF365 x MODIS — Full Correlation Analysis (3 methods x 6 index-pairs x 4 views)

3 alignment methods x 6 index-pairs (3 HF365 indices x 2 MODIS
variables) x 4 views = **72 correlation numbers**, plus all 72 as
individual charts (grouped 4-per-pair, 6-pairs-per-method).

## Bug fix in this version
The QC-weighted method (Method 3) previously applied its weight map
directly to the **raw packed `FparLai_QC` byte** -- but that byte packs
five different sub-fields (MODLAND flag, sensor, dead-detector flag,
cloud state, and the actual SCF_QC quality level) into one number. A
weight map like `{0: 1.0, 2: 0.75}` applied straight to the raw byte
was accidentally keying off *combinations of unrelated bits*, not
actual quality -- e.g. raw byte value 2 corresponds to
sensor=Aqua+SCF_QC=Best, not a distinct quality tier. This silently
mis-weighted roughly a third of the data (all "Good, with saturation"
observations landed in the near-zero default weight bucket purely
because of which sensor or cloud-flag bits they happened to have).

**Fix:** decode the real SCF_QC level first via `(byte >> 5) & 0b111`
(bits 5-7), *then* map that decoded level to a weight. This is the same
decode function validated in `01-data-understanding`.


In [ ]:
import os
import re
import glob
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from numpy.polynomial import polynomial as P

sns.set_style("whitegrid")

# vvv EDIT THESE to match your folders vvv
HF365_DIR = "."
MODIS_DIR = "modis_downloads"
# ^^^ EDIT THESE to match your folders vvv

HF365_FILES = {
    "EVI2": "hf365-01-hls-evi2.csv",
    "NDVI": "hf365-02-hls-ndvi.csv",
    "NIRv": "hf365-03-hls-nirv.csv",
}
MODIS_VARS = ["Lai_500m", "Fpar_500m"]
NEAREST_TOLERANCE_DAYS = 4


## 1. Load HF365

In [ ]:
hf365_wide = {}
for name, fname in HF365_FILES.items():
    df = pd.read_csv(os.path.join(HF365_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    hf365_wide[name] = df
    print(f"{name}: {df.shape[0]} dates x {df.shape[1]} plots")


## 2. Load every MODIS layer

In [ ]:
FILE_PATTERN_TEMPLATE = r"MCD15A2H\.061_{layer}_(\d{{8}})T\d+_aid(\d+)"
SCALE = {"Lai_500m": 0.1, "Fpar_500m": 0.01, "LaiStdDev_500m": 0.01, "FparStdDev_500m": 0.01}
FILL_VALUES = {249, 250, 251, 252, 253, 254, 255}

all_modis_files = glob.glob(os.path.join(MODIS_DIR, "**", "*.tif"), recursive=True)
print(f"Total .tif files found in {MODIS_DIR}: {len(all_modis_files)}")

def load_modis_layer(layer_name, apply_scale=True):
    pattern = re.compile(FILE_PATTERN_TEMPLATE.format(layer=re.escape(layer_name)))
    files = [p for p in all_modis_files if layer_name in os.path.basename(p)]
    records = []
    for path in files:
        fname = os.path.basename(path)
        m = pattern.search(fname)
        if not m:
            continue
        date_str, pixel_num = m.groups()
        date = pd.to_datetime(date_str, format="%Y%m%d")
        pixel_id = f"pixel_{int(pixel_num)}"
        with rasterio.open(path) as src:
            arr = src.read(1)
            raw = arr[1, 1] if arr.shape == (2, 2) else np.nan
        if layer_name == "FparLai_QC":
            value = raw  # keep raw byte -- decoded separately in section 4
        else:
            value = np.nan if raw in FILL_VALUES else raw * SCALE.get(layer_name, 1.0)
        records.append({"date": date, "pixel": pixel_id, "value": value})
    long_df = pd.DataFrame(records)
    if long_df.empty:
        raise FileNotFoundError(f"No files found for layer '{layer_name}' in {MODIS_DIR}")
    wide = long_df.pivot_table(index="date", columns="pixel", values="value", aggfunc="first")
    return wide.sort_index()

modis_layers = {}
for layer in ["Lai_500m", "Fpar_500m", "FparLai_QC", "LaiStdDev_500m", "FparStdDev_500m"]:
    modis_layers[layer] = load_modis_layer(layer)
    print(f"{layer}: {modis_layers[layer].shape[0]} dates x {modis_layers[layer].shape[1]} pixels")


## 3. Build the true MODIS windows

In [ ]:
modis_dates_sorted = modis_layers["Lai_500m"].index.sort_values()
windows = [(modis_dates_sorted[i], modis_dates_sorted[i+1] - pd.Timedelta(days=1))
           for i in range(len(modis_dates_sorted) - 1)]
print(f"Built {len(windows)} true MODIS windows "
      f"(from {modis_dates_sorted[0].date()} to {modis_dates_sorted[-1].date()})")


## 4. QC -> confidence weight (FIXED: decodes the real SCF_QC level first)

In [ ]:
def decode_scf_qc(qc_byte):
    '''SCF_QC 'goodness' level lives in bits 5-7 of the packed QC byte.'''
    if pd.isna(qc_byte):
        return np.nan
    return (int(qc_byte) >> 5) & 0b111

# Weight by the DECODED level, not the raw byte
SCF_QC_WEIGHT = {0: 1.00, 1: 0.85, 2: 0.50, 3: 0.20, 4: 0.05}

scf_qc_df = modis_layers["FparLai_QC"].map(decode_scf_qc)
qc_weight_df = scf_qc_df.map(lambda v: SCF_QC_WEIGHT.get(v, 0.05) if pd.notna(v) else np.nan)

print("Decoded SCF_QC level distribution (all pixels/dates):")
print(scf_qc_df.stack().value_counts().sort_index())

STDDEV_EPS = 0.05

def combined_weight_for(modis_var):
    stddev_layer = "LaiStdDev_500m" if modis_var == "Lai_500m" else "FparStdDev_500m"
    inv_var = 1.0 / (modis_layers[stddev_layer].clip(lower=STDDEV_EPS) ** 2)
    return qc_weight_df.reindex_like(inv_var) * inv_var

combined_weights = {var: combined_weight_for(var) for var in MODIS_VARS}
for var, w in combined_weights.items():
    print(f"Combined weight for {var}: mean={w.stack().mean():.3f}, "
          f"non-null cells={w.notna().sum().sum()} / {w.size}")


## 5. Alignment functions for all 3 methods

All three align by **date value**, not positional index -- this avoids
a subtle bug where mismatched-length frames (e.g. partial QC/StdDev
downloads) get misaligned by position instead of by date.

In [ ]:
def align_nearest(hf_df, modis_wide, tolerance_days=NEAREST_TOLERANCE_DAYS):
    modis_dates_arr = modis_wide.index.to_numpy()
    matched_hf, matched_px, matched_dates = [], [], []
    for d in hf_df.index:
        diffs = np.abs((modis_dates_arr - np.datetime64(d)).astype("timedelta64[D]").astype(int))
        idx = diffs.argmin()
        if diffs[idx] <= tolerance_days:
            matched_hf.append(hf_df.loc[d])
            matched_px.append(modis_wide.iloc[idx])
            matched_dates.append(d)
    if not matched_dates:
        return pd.DataFrame(columns=hf_df.columns), pd.DataFrame(columns=modis_wide.columns)
    return (pd.DataFrame(matched_hf, index=matched_dates), pd.DataFrame(matched_px, index=matched_dates))


def align_window_containment(hf_df, modis_wide, windows):
    matched_hf, matched_px, matched_dates = [], [], []
    for w_start, w_end in windows:
        in_window = hf_df.loc[(hf_df.index >= w_start) & (hf_df.index <= w_end)]
        if in_window.empty or w_start not in modis_wide.index:
            continue
        matched_hf.append(in_window.mean(axis=0))
        matched_px.append(modis_wide.loc[w_start])
        matched_dates.append(w_start)
    if not matched_dates:
        return pd.DataFrame(columns=hf_df.columns), pd.DataFrame(columns=modis_wide.columns)
    return (pd.DataFrame(matched_hf, index=matched_dates), pd.DataFrame(matched_px, index=matched_dates))


def align_weighted(hf_df, modis_wide, weight_wide, windows):
    hf_a, px_a = align_window_containment(hf_df, modis_wide, windows)
    w_a = weight_wide.reindex(px_a.index)
    return hf_a, px_a, w_a


## 6. Correlation scoring functions

In [ ]:
def score_views(hf_aligned, px_aligned):
    corr_matrix = pd.DataFrame(index=hf_aligned.columns, columns=px_aligned.columns, dtype=float)
    for plot in hf_aligned.columns:
        for pixel in px_aligned.columns:
            corr_matrix.loc[plot, pixel] = hf_aligned[plot].corr(px_aligned[pixel])
    avg_plot = hf_aligned.mean(axis=1)
    avg_pixel = px_aligned.mean(axis=1)
    return {
        "plot_vs_pixel_matrix": corr_matrix,
        "avg_plot_vs_pixel": px_aligned.apply(lambda col: avg_plot.corr(col)),
        "plot_vs_avg_pixel": hf_aligned.apply(lambda col: col.corr(avg_pixel)),
        "avg_plot_vs_avg_pixel": avg_plot.corr(avg_pixel),
    }


def weighted_pearson(x, y, w):
    x, y, w = (np.asarray(a, dtype=float) for a in (x, y, w))
    mask = ~(np.isnan(x) | np.isnan(y) | np.isnan(w)) & (w > 0)
    x, y, w = x[mask], y[mask], w[mask]
    if len(x) < 2 or w.sum() == 0:
        return np.nan
    x_bar, y_bar = np.average(x, weights=w), np.average(y, weights=w)
    cov = np.sum(w * (x - x_bar) * (y - y_bar))
    var_x, var_y = np.sum(w * (x - x_bar) ** 2), np.sum(w * (y - y_bar) ** 2)
    return cov / np.sqrt(var_x * var_y) if var_x > 0 and var_y > 0 else np.nan


def score_views_weighted(hf_aligned, px_aligned, w_aligned):
    avg_plot = hf_aligned.mean(axis=1)
    avg_pixel = px_aligned.mean(axis=1)
    avg_weight = w_aligned.mean(axis=1)
    corr_matrix = pd.DataFrame(index=hf_aligned.columns, columns=px_aligned.columns, dtype=float)
    for plot in hf_aligned.columns:
        for pixel in px_aligned.columns:
            w = w_aligned[pixel] if pixel in w_aligned.columns else avg_weight
            corr_matrix.loc[plot, pixel] = weighted_pearson(hf_aligned[plot], px_aligned[pixel], w)
    return {
        "plot_vs_pixel_matrix": corr_matrix,
        "avg_plot_vs_pixel": pd.Series({p: weighted_pearson(avg_plot, px_aligned[p],
                                        w_aligned[p] if p in w_aligned.columns else avg_weight)
                                        for p in px_aligned.columns}),
        "plot_vs_avg_pixel": pd.Series({p: weighted_pearson(hf_aligned[p], avg_pixel, avg_weight)
                                        for p in hf_aligned.columns}),
        "avg_plot_vs_avg_pixel": weighted_pearson(avg_plot, avg_pixel, avg_weight),
    }


## 7. Run all 72 combinations

In [ ]:
all_results = {}
all_aligned = {}

for hf_name, hf_df in hf365_wide.items():
    for modis_var in MODIS_VARS:
        modis_wide = modis_layers[modis_var]

        hf_a1, px_a1 = align_nearest(hf_df, modis_wide)
        all_results[("nearest_incl_zero", hf_name, modis_var)] = score_views(hf_a1, px_a1) if len(hf_a1) >= 2 else None
        all_aligned[("nearest_incl_zero", hf_name, modis_var)] = (hf_a1, px_a1, None)

        hf_a2, px_a2 = align_window_containment(hf_df, modis_wide, windows)
        all_results[("window_excl_zero", hf_name, modis_var)] = score_views(hf_a2, px_a2) if len(hf_a2) >= 2 else None
        all_aligned[("window_excl_zero", hf_name, modis_var)] = (hf_a2, px_a2, None)

        hf_a3, px_a3, w_a3 = align_weighted(hf_df, modis_wide, combined_weights[modis_var], windows)
        all_results[("qc_stddev_weighted", hf_name, modis_var)] = score_views_weighted(hf_a3, px_a3, w_a3) if len(hf_a3) >= 2 else None
        all_aligned[("qc_stddev_weighted", hf_name, modis_var)] = (hf_a3, px_a3, w_a3)

print(f"Total combinations run: {len(all_results)} (expect 3 x 3 x 2 = 18, each with 4 views = 72 numbers)")


## 8. Summary table -- all 72 numbers

In [ ]:
summary_rows = []
for (method, hf_name, modis_var), res in all_results.items():
    if res is None:
        summary_rows.append(dict(method=method, hf_index=hf_name, modis_var=modis_var,
                                  mean_plot_vs_pixel=np.nan, avg_plot_vs_pixel=np.nan,
                                  plot_vs_avg_pixel=np.nan, avg_plot_vs_avg_pixel=np.nan))
        continue
    summary_rows.append(dict(
        method=method, hf_index=hf_name, modis_var=modis_var,
        mean_plot_vs_pixel=res["plot_vs_pixel_matrix"].values.astype(float).mean(),
        avg_plot_vs_pixel=res["avg_plot_vs_pixel"].mean(),
        plot_vs_avg_pixel=res["plot_vs_avg_pixel"].mean(),
        avg_plot_vs_avg_pixel=res["avg_plot_vs_avg_pixel"],
    ))

summary_df = pd.DataFrame(summary_rows).round(3)
summary_df


## 9. Method comparison

In [ ]:
method_comparison = summary_df.pivot_table(
    index=["hf_index", "modis_var"], columns="method", values="avg_plot_vs_avg_pixel"
)[["nearest_incl_zero", "window_excl_zero", "qc_stddev_weighted"]]
method_comparison


## 10. All 72 charts, grouped 4-per-pair, 6-pairs-per-method

In [ ]:
METHOD_TITLES = {
    "nearest_incl_zero": "Method 1: Nearest-date (includes zero-windows)",
    "window_excl_zero": "Method 2: Strict window-containment (excludes zero-windows)",
    "qc_stddev_weighted": "Method 3: QC + StdDev weighted (fixed SCF_QC decode)",
}
INDEX_PAIRS = [(hf, mv) for hf in HF365_FILES for mv in MODIS_VARS]

def plot_four_views(method, hf_name, modis_var):
    res = all_results[(method, hf_name, modis_var)]
    hf_a, px_a, w_a = all_aligned[(method, hf_name, modis_var)]
    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
    fig.suptitle(f"{hf_name} vs {modis_var}  --  {METHOD_TITLES[method]}", fontsize=12, y=1.05)

    if res is None or len(hf_a) < 2:
        for ax in axes:
            ax.text(0.5, 0.5, "Not enough matched points", ha="center", va="center", transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout(); plt.show(); return

    ax = axes[0]
    sns.heatmap(res["plot_vs_pixel_matrix"].astype(float), cmap="RdBu_r", vmin=-1, vmax=1,
                center=0, cbar_kws={"label": "r"}, ax=ax, yticklabels=True)
    ax.set_title("[1] Plot vs Pixel"); ax.tick_params(axis="y", labelsize=5)

    ax = axes[1]
    vals = res["avg_plot_vs_pixel"]
    ax.bar(vals.index, vals.values, color="steelblue"); ax.set_ylim(0, 1)
    ax.set_title("[2] Avg-plot vs each Pixel"); ax.tick_params(axis="x", rotation=45)

    ax = axes[2]
    vals = res["plot_vs_avg_pixel"]
    ax.bar(range(len(vals)), vals.values, color="seagreen")
    ax.set_xticks(range(len(vals))); ax.set_xticklabels(vals.index, rotation=90, fontsize=5)
    ax.set_ylim(0, 1); ax.set_title("[3] Each Plot vs Avg-pixel")

    ax = axes[3]
    avg_plot, avg_pixel = hf_a.mean(axis=1), px_a.mean(axis=1)
    ax.scatter(avg_plot, avg_pixel, s=18, alpha=0.6, color="darkorange")
    mask = avg_plot.notna() & avg_pixel.notna()
    if mask.sum() >= 2:
        coefs = P.polyfit(avg_plot[mask], avg_pixel[mask], 1)
        xs = np.linspace(avg_plot[mask].min(), avg_plot[mask].max(), 50)
        ax.plot(xs, P.polyval(xs, coefs), color="black", linewidth=1)
    ax.set_title(f"[4] Avg-plot vs Avg-pixel (r={res['avg_plot_vs_avg_pixel']:.3f})")
    plt.tight_layout(); plt.show()


### Method 1: Nearest-date -- 6 index-pairs

In [ ]:
for hf_name, modis_var in INDEX_PAIRS:
    plot_four_views("nearest_incl_zero", hf_name, modis_var)


### Method 2: Window-containment -- 6 index-pairs

In [ ]:
for hf_name, modis_var in INDEX_PAIRS:
    plot_four_views("window_excl_zero", hf_name, modis_var)


### Method 3: QC + StdDev weighted (fixed) -- 6 index-pairs

In [ ]:
for hf_name, modis_var in INDEX_PAIRS:
    plot_four_views("qc_stddev_weighted", hf_name, modis_var)
